In [4]:
# Cell 1
# Mục đích: định nghĩa lại kết nối và các hàm dùng chung (notebook mới phải định nghĩa lại).
import os
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()                                  # đọc file .env vào biến môi trường
PROJECT = os.getenv("GCP_PROJECT", "")         # os.getenv chỉ đọc biến đã được nạp
LOCATION = os.getenv("BQ_LOCATION", "US")      # "US" là giá trị mặc định
print(PROJECT, LOCATION)                       # kiểm tra không rỗng trước khi đi tiếp

client = bigquery.Client(project=PROJECT, location=LOCATION)   # kênh kết nối tới BigQuery


def mb(n):
    # Đổi byte sang MB; trả None nếu n là None để lỗi đọc quá sớm lộ ra.
    return None if n is None else round(n / 1024 / 1024, 2)


def check(sql):
    # Dry run: chỉ kiểm tra cú pháp và ước lượng byte, không chạy thật, không tốn tiền.
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")


def stats(sql, max_mb=100):
    # Chạy thật, tắt cache, đặt trần byte bị tính tiền (max_mb đổi sang byte).
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,
    )
    job = client.query(sql, job_config=cfg)        # gửi job, không chờ
    df = job.result().to_dataframe()               # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df                                      # thiếu dòng này thì df ở ngoài là None

bq-learning-510104 US


In [5]:
# Cell 2a
# Mục đích: bảng order_items, so NULL trước và sau SAFE_CAST cho 4 cột sắp ép kiểu.
sql = f"""
SELECT
    COUNT(*) AS n,
    COUNTIF(order_item_id IS NULL) AS item_raw,
    COUNTIF(SAFE_CAST(order_item_id AS INT64) IS NULL) AS item_cast,
    COUNTIF(shipping_limit_date IS NULL) AS ship_raw,
    COUNTIF(SAFE_CAST(shipping_limit_date AS TIMESTAMP) IS NULL) AS ship_cast,
    COUNTIF(price IS NULL) AS price_raw,
    COUNTIF(SAFE_CAST(price AS NUMERIC) IS NULL) AS price_cast,
    COUNTIF(freight_value IS NULL) AS freight_raw,
    COUNTIF(SAFE_CAST(freight_value AS NUMERIC) IS NULL) AS freight_cast
FROM `{PROJECT}.olist_raw.order_items`
"""
check(sql)
df = stats(sql)
df.T

Sẽ quét: 4.11 MB
processed: 4.11 MB | billed: 10.0 MB


,0
n,112650
item_raw,0
item_cast,0
ship_raw,0
ship_cast,0
price_raw,0
price_cast,0
freight_raw,0
freight_cast,0


In [6]:
# Cell 2b
# Mục đích: bảng order_payments, so NULL trước và sau SAFE_CAST cho 3 cột.
sql = f"""
SELECT
    COUNT(*) AS n,
    COUNTIF(payment_sequential IS NULL) AS seq_raw,
    COUNTIF(SAFE_CAST(payment_sequential AS INT64) IS NULL) AS seq_cast,
    COUNTIF(payment_installments IS NULL) AS inst_raw,
    COUNTIF(SAFE_CAST(payment_installments AS INT64) IS NULL) AS inst_cast,
    COUNTIF(payment_value IS NULL) AS value_raw,
    COUNTIF(SAFE_CAST(payment_value AS NUMERIC) IS NULL) AS value_cast
FROM `{PROJECT}.olist_raw.order_payments`
"""
check(sql)
df = stats(sql)
df.T

Sẽ quét: 1.34 MB
processed: 1.34 MB | billed: 10.0 MB


,0
n,103886
seq_raw,0
seq_cast,0
inst_raw,0
inst_cast,0
value_raw,0
value_cast,0


In [7]:
# Cell 2c
# Mục đích: bảng order_reviews, so NULL trước và sau SAFE_CAST cho 3 cột.
sql = f"""
SELECT
    COUNT(*) AS n,
    COUNTIF(review_score IS NULL) AS score_raw,
    COUNTIF(SAFE_CAST(review_score AS INT64) IS NULL) AS score_cast,
    COUNTIF(review_creation_date IS NULL) AS created_raw,
    COUNTIF(SAFE_CAST(review_creation_date AS TIMESTAMP) IS NULL) AS created_cast,
    COUNTIF(review_answer_timestamp IS NULL) AS answered_raw,
    COUNTIF(SAFE_CAST(review_answer_timestamp AS TIMESTAMP) IS NULL) AS answered_cast
FROM `{PROJECT}.olist_raw.order_reviews`
"""
check(sql)
df = stats(sql)
df.T

Sẽ quét: 4.26 MB
processed: 4.26 MB | billed: 10.0 MB


,0
n,99224
score_raw,0
score_cast,0
created_raw,0
created_cast,0
answered_raw,0
answered_cast,0


In [8]:
# Cell 2d
# Mục đích: bảng products, so NULL trước và sau SAFE_CAST cho 7 cột số (tên giữ nguyên lỗi chính tả lenght của nguồn).
sql = f"""
SELECT
    COUNT(*) AS n,
    COUNTIF(product_name_lenght IS NULL) AS namelen_raw,
    COUNTIF(SAFE_CAST(product_name_lenght AS INT64) IS NULL) AS namelen_cast,
    COUNTIF(product_description_lenght IS NULL) AS desclen_raw,
    COUNTIF(SAFE_CAST(product_description_lenght AS INT64) IS NULL) AS desclen_cast,
    COUNTIF(product_photos_qty IS NULL) AS photos_raw,
    COUNTIF(SAFE_CAST(product_photos_qty AS INT64) IS NULL) AS photos_cast,
    COUNTIF(product_weight_g IS NULL) AS weight_raw,
    COUNTIF(SAFE_CAST(product_weight_g AS INT64) IS NULL) AS weight_cast,
    COUNTIF(product_length_cm IS NULL) AS length_raw,
    COUNTIF(SAFE_CAST(product_length_cm AS INT64) IS NULL) AS length_cast,
    COUNTIF(product_height_cm IS NULL) AS height_raw,
    COUNTIF(SAFE_CAST(product_height_cm AS INT64) IS NULL) AS height_cast,
    COUNTIF(product_width_cm IS NULL) AS width_raw,
    COUNTIF(SAFE_CAST(product_width_cm AS INT64) IS NULL) AS width_cast
FROM `{PROJECT}.olist_raw.products`
"""
check(sql)
df = stats(sql)
df.T

Sẽ quét: 0.92 MB
processed: 0.92 MB | billed: 10.0 MB


,0
n,32951
namelen_raw,610
namelen_cast,610
desclen_raw,610
desclen_cast,610
photos_raw,610
photos_cast,610
weight_raw,2
weight_cast,2
length_raw,2


In [ ]:
# Cell 4
# Mục đích: xem tên cột thật của bảng translation trước khi viết staging cho nó.
t = client.get_table(f"{PROJECT}.olist_raw.product_category_translation")
for f in t.schema:                 # mỗi vòng gán một cột vào f
    print(f.name, f.field_type)

product_category_name STRING
product_category_name_english STRING


In [10]:
# Cell 5
# Mục đích: so số dòng mỗi view staging với số dòng đếm từ file CSV (nguồn ngoài BigQuery).
EXPECTED = {
    "stg_olist__customers": 99441,
    "stg_olist__sellers": 3095,
    "stg_olist__orders": 99441,
    "stg_olist__order_items": 112650,
    "stg_olist__order_payments": 103886,
    "stg_olist__order_reviews": 99224,
    "stg_olist__products": 32951,
    "stg_olist__product_category_translation": 71,
}
print("Số view dự định:", len(EXPECTED))     # bắt trường hợp dictionary rỗng

for view, n_csv in EXPECTED.items():          # mỗi vòng gán một cặp (tên view, số dòng CSV)
    sql = f"SELECT COUNT(*) AS so_dong FROM `{PROJECT}.olist_dbt_dev.{view}`"
    df = stats(sql)                           # df chỉ có 1 dòng (kết quả COUNT)
    n_view = int(df["so_dong"].iloc[0])       # số dòng của view nằm trong ô so_dong
    print(f"{view:45s} view {n_view:>9,} | csv {n_csv:>9,} | {'OK' if n_view == n_csv else 'LỆCH'}")

Số view dự định: 8
processed: 0.0 MB | billed: 0.0 MB
stg_olist__customers                          view    99,441 | csv    99,441 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__sellers                            view     3,095 | csv     3,095 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__orders                             view    99,441 | csv    99,441 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__order_items                        view   112,650 | csv   112,650 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__order_payments                     view   103,886 | csv   103,886 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__order_reviews                      view    99,224 | csv    99,224 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__products                           view    32,951 | csv    32,951 | OK
processed: 0.0 MB | billed: 0.0 MB
stg_olist__product_category_translation       view        71 | csv        71 | OK


In [11]:
# Cell 7
# Mục đích: xem 10 mã zip có nhiều dòng nhất, mỗi zip có mấy tên thành phố, mấy bang, tọa độ lệch bao nhiêu.
sql = f"""
SELECT
    geolocation_zip_code_prefix AS zip,
    COUNT(*) AS n,
    COUNT(DISTINCT geolocation_city) AS n_city,
    COUNT(DISTINCT geolocation_state) AS n_state,
    MAX(SAFE_CAST(geolocation_lat AS FLOAT64)) - MIN(SAFE_CAST(geolocation_lat AS FLOAT64)) AS lat_range
FROM `{PROJECT}.olist_raw.geolocation`
GROUP BY zip
ORDER BY n DESC
LIMIT 10
"""
check(sql)
df = stats(sql)
df

Sẽ quét: 42.06 MB
processed: 42.06 MB | billed: 43.0 MB


,zip,n,n_city,n_state,lat_range
0,24220,1146,2,1,0.012119
1,24230,1102,2,1,1.700545
2,38400,965,2,1,0.048244
3,35500,907,2,1,0.063854
4,11680,879,1,1,0.214494
5,22631,832,1,1,0.010619
6,30140,810,1,1,0.017234
7,11740,788,2,1,0.159523
8,38408,773,2,1,0.033622
9,28970,743,1,1,0.211341


In [12]:
# Cell 8
# Mục đích: đếm trên toàn bảng có bao nhiêu zip có nhiều tên thành phố, nhiều bang, tọa độ lệch rộng.
sql = f"""
WITH z AS (
    SELECT
        geolocation_zip_code_prefix AS zip,
        COUNT(*) AS n,
        COUNT(DISTINCT geolocation_city) AS n_city,
        COUNT(DISTINCT geolocation_state) AS n_state,
        MAX(SAFE_CAST(geolocation_lat AS FLOAT64)) - MIN(SAFE_CAST(geolocation_lat AS FLOAT64)) AS lat_range
    FROM `{PROJECT}.olist_raw.geolocation`
    GROUP BY zip
)
SELECT
    COUNT(*) AS so_zip,
    COUNTIF(n_city > 1) AS zip_nhieu_city,
    COUNTIF(n_state > 1) AS zip_nhieu_state,
    COUNTIF(lat_range > 0.1) AS zip_lat_rong,
    MAX(n) AS n_max
FROM z
"""
check(sql)
df = stats(sql)
df.T          # xoay dòng thành cột để dễ đọc

Sẽ quét: 42.06 MB
processed: 42.06 MB | billed: 43.0 MB


,0
so_zip,19015
zip_nhieu_city,8556
zip_nhieu_state,8
zip_lat_rong,2001
n_max,1146


In [14]:
# Cell 5b
# Mục đích: tự nhìn từng lớp của df["so_dong"].iloc[0]; tự tạo df nhỏ để không phụ thuộc cell khác.
sql = f"SELECT COUNT(*) AS so_dong FROM `{PROJECT}.olist_dbt_dev.stg_olist__orders`"
df = stats(sql)                        # df mới: 1 dòng, 1 cột so_dong
print(df)                              # cả bảng: 1 dòng, 1 cột
print(df["so_dong"])                   # chỉ cột so_dong (vẫn là một dãy)
print(df["so_dong"].iloc[0])           # phần tử đầu tiên: một con số
print(type(df["so_dong"].iloc[0]), type(int(df["so_dong"].iloc[0])))   # kiểu numpy và kiểu int

processed: 0.0 MB | billed: 0.0 MB
   so_dong
0    99441
0    99441
Name: so_dong, dtype: Int64
99441
<class 'numpy.int64'> <class 'int'>
